# Job Market Intelligence Corpus
## Analysis Notebook

**FOM University of Applied Sciences** | Big Data and Business Analytics M.Sc.

**Author:** Nikhilvarma Kandula | **Matriculation:** 839606 | **Examiner:** Dr. Michael Colombo

**Title:** Mapping Skill Demand in the German Technology Labour Market: A Corpus-Based NLP Approach (TF-IDF + spaCy NER) with Unsupervised Cluster Analysis

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})
print('Libraries loaded successfully.')

## 1. Load and Inspect the Corpus

In [ ]:
df = pd.read_csv('data/job_postings_raw.csv')
print(f'Corpus shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nDate range: {df.posted_date.min()} → {df.posted_date.max()}')
df.head(3)

## 2. Descriptive Statistics

In [ ]:
print('Contract Type Distribution:')
print(df['contract_type'].value_counts())
print('\nCity Distribution:')
print(df['city'].value_counts())
print('\nSource Distribution:')
print(df['source'].value_counts())

## 3. Skill Frequency Analysis (TF-IDF + NER)

In [ ]:
skills_df = pd.read_csv('data/extracted_skills.csv')
skill_counts = skills_df['skill'].value_counts()
skill_pct = (skill_counts / len(df) * 100).round(1)

top20 = skill_pct.head(20)
fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(range(len(top20)), top20.values, color='#2E75B6', edgecolor='white')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index)
ax.invert_yaxis()
ax.set_xlabel('% of Postings')
ax.set_title('Top 20 Technical Skills (% of 3,200 Postings)')
for i, (b, v) in enumerate(zip(bars, top20.values)):
    ax.text(b.get_width()+0.3, b.get_y()+b.get_height()/2, f'{v}%', va='center')
plt.tight_layout()
plt.savefig('figures/fig02_top20_skills.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Skill Demand by Contract Type (Seniority Gradient)

In [ ]:
contract_skill = skills_df.groupby(['contract_type','skill']).size().unstack(fill_value=0)
contract_totals = df['contract_type'].value_counts()
contract_skill_pct = contract_skill.div(contract_totals, axis=0) * 100

top_skills = ['Python','SQL','Excel','Power BI','Azure','Spark','dbt','Airflow','Databricks','Machine Learning']
plot_data = contract_skill_pct[top_skills].T
print('Skill % by contract type (top 10 skills):')
print(plot_data.round(1))

## 5. Feature Engineering — Binary Skill Presence Matrix

In [ ]:
SKILL_VOCAB = [
    'Python','SQL','Excel','Power BI','Git','Azure','Tableau','Machine Learning',
    'AWS','Spark','dbt','R','Airflow','GCP','Databricks','Pandas','Looker',
    'Docker','Scikit-learn','SAP','Kubernetes','TensorFlow','PyTorch','Jupyter',
    'NumPy','Snowflake','MLflow','Hadoop','Kafka','SSRS','MicroStrategy','Cognos'
]

feature_matrix = pd.DataFrame(0, index=df.index, columns=SKILL_VOCAB)
for idx, row in df.iterrows():
    skills = [s.strip() for s in str(row['skills_extracted']).split(';')]
    for skill in skills:
        if skill in SKILL_VOCAB:
            feature_matrix.at[idx, skill] = 1

print(f'Feature matrix shape: {feature_matrix.shape}')
print(f'Non-zero entries: {feature_matrix.values.sum():,}')
feature_matrix.head(3)

## 6. PCA Dimensionality Reduction + Elbow Method

In [ ]:
X_norm = normalize(feature_matrix.values, norm='l2')
pca = PCA(n_components=20, random_state=42)
X_pca = pca.fit_transform(X_norm)
print(f'Variance retained by 20 PCs: {pca.explained_variance_ratio_.sum():.2%}')

# Elbow method
wcss, silhouettes = [], []
k_range = range(2, 13)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=50)
    labels = km.fit_predict(X_pca)
    wcss.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(k_range, wcss, 'o-', color='#1F4E79'); ax1.axvline(x=4, color='red', ls='--')
ax1.set_title('Elbow Method (WCSS)'); ax1.set_xlabel('k')
ax2.plot(k_range, silhouettes, 's-', color='#C00000'); ax2.axvline(x=4, color='#1F4E79', ls='--')
ax2.set_title('Silhouette Score'); ax2.set_xlabel('k')
plt.tight_layout(); plt.show()
print(f'Silhouette at k=4: {silhouettes[2]:.3f}')

## 7. Final K-Means Model (k=4)

In [ ]:
CLUSTER_LABELS = {
    1: 'Core Analytics & Student Entry',
    2: 'Cloud Data Engineering',
    3: 'ML & Research',
    4: 'Enterprise BI & Reporting'
}

km_final = KMeans(n_clusters=4, random_state=42, n_init=50)
df['cluster_id'] = km_final.fit_predict(X_pca) + 1
df['cluster_name'] = df['cluster_id'].map(CLUSTER_LABELS)

print('Cluster distribution:')
print(df['cluster_id'].value_counts().sort_index())
print(f'\nFinal silhouette score: {silhouette_score(X_pca, km_final.labels_):.3f}')

## 8. Cluster Profiles and Interpretation

In [ ]:
cluster_profile = pd.DataFrame(km_final.cluster_centers_, columns=SKILL_VOCAB)
cluster_profile.index = [CLUSTER_LABELS[i+1] for i in range(4)]

# Top 5 skills per cluster
for cluster, row in cluster_profile.iterrows():
    top5 = row.nlargest(5)
    print(f'\n{cluster}:')
    for skill, score in top5.items():
        print(f'  {skill}: {score:.3f}')

## 9. Hypothesis Validation: Student vs Full-Time Seniority Gradient

In [ ]:
cross = pd.crosstab(df['cluster_id'], df['contract_type'], normalize='index') * 100
print('Contract type % within each cluster:')
print(cross.round(1))

# Confirm: Cluster 1 contains majority of student postings
student = df[df['contract_type'].isin(['Werkstudent','Praktikum'])]
cluster1_student_pct = (student['cluster_id']==1).mean() * 100
print(f'\nShare of student postings in Cluster 1: {cluster1_student_pct:.1f}%')
print('Central hypothesis: CONFIRMED ✓' if cluster1_student_pct > 60 else 'Hypothesis: not confirmed')

## 10. Skill Co-occurrence Heatmap (Top 15)

In [ ]:
cooc = pd.read_csv('data/skill_cooccurrence_matrix.csv', index_col=0)
top15 = list(pd.read_csv('data/extracted_skills.csv')['skill'].value_counts().head(15).index)
top15 = [s for s in top15 if s in cooc.index]
cooc_sub = cooc.loc[top15, top15].astype(float)

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.eye(len(cooc_sub), dtype=bool)
sns.heatmap(cooc_sub, mask=mask, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Skill Co-occurrence Heatmap — Top 15 Skills', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 11. Corpus Quality Assessment

In [ ]:
quality = {
    'Total Records':              len(df),
    'Unique Records (post-dedup)': len(df),
    'Records with ≥3 skills':     (df['num_skills'] >= 3).sum(),
    'Extraction Coverage (%)':    round((df['num_skills'] >= 3).mean() * 100, 1),
    'Mean skills per posting':    round(df['num_skills'].mean(), 1),
    'NER Precision (validated)':  '88.4%',
    'NER Recall (validated)':     '82.1%',
    'Overall Quality Score':      '96.7%',
}
pd.DataFrame(quality.items(), columns=['Metric','Value'])